In [1]:
import numpy as np
import json
from pathlib import Path

GRAPH_SPEC_DIR = Path("outputs/graph_spectrum")

node_order = np.load(GRAPH_SPEC_DIR / "node_order.npy").astype(int).tolist()

print("Number of graph nodes:", len(node_order))
print("First 10 node IDs:", node_order[:10])

Number of graph nodes: 1570
First 10 node IDs: [512, 513, 514, 516, 593, 594, 595, 596, 597, 598]


In [6]:
import pickle
import pandas as pd

PKL_PATH = Path(r"C:\Users\USER\Documents\GitHub\datasets\simbarca\all_agg\agg_timeseries_000.pkl")

with open(PKL_PATH, "rb") as f:
    data = pickle.load(f)

pred_vtime = data["pred_vtime"].copy()

# Make sure column names are ints
pred_vtime.columns = pred_vtime.columns.astype(int)

graph_nodes = set(node_order)
pickle_nodes = set(pred_vtime.columns.tolist())

missing_in_pickle = sorted(graph_nodes - pickle_nodes)
extra_in_pickle = sorted(pickle_nodes - graph_nodes)

print("pred_vtime shape:", pred_vtime.shape)
print("Graph node count:", len(graph_nodes))
print("Pickle node count:", len(pickle_nodes))
print("Missing in pickle:", len(missing_in_pickle))
print("Extra in pickle:", len(extra_in_pickle))

if len(missing_in_pickle) > 0:
    print("First 10 missing:", missing_in_pickle[:10])

if len(extra_in_pickle) > 0:
    print("First 10 extra:", extra_in_pickle[:10])

print("Exact same node set:", graph_nodes == pickle_nodes)
print("Exact same order already:", pred_vtime.columns.tolist() == node_order)

pred_vtime shape: (54, 1570)
Graph node count: 1570
Pickle node count: 1570
Missing in pickle: 0
Extra in pickle: 0
Exact same node set: True
Exact same order already: True


In [7]:
import pickle
from pathlib import Path
import pandas as pd

PICKLE_DIR = Path(r"C:\Users\USER\Documents\GitHub\datasets\simbarca\all_agg")
pkl_files = sorted(PICKLE_DIR.glob("*.pkl"))

print("Number of pickle files found:", len(pkl_files))

audit_rows = []

for pkl_path in pkl_files:
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    pred_vtime = data["pred_vtime"].copy()
    pred_vtime.columns = pred_vtime.columns.astype(int)

    graph_nodes = set(node_order)
    pickle_nodes = set(pred_vtime.columns.tolist())

    missing_in_pickle = len(graph_nodes - pickle_nodes)
    extra_in_pickle = len(pickle_nodes - graph_nodes)
    exact_same_set = (graph_nodes == pickle_nodes)
    exact_same_order = (pred_vtime.columns.tolist() == node_order)

    audit_rows.append({
        "file": pkl_path.name,
        "shape_rows": pred_vtime.shape[0],
        "shape_cols": pred_vtime.shape[1],
        "same_node_set": exact_same_set,
        "same_order": exact_same_order,
        "missing_nodes": missing_in_pickle,
        "extra_nodes": extra_in_pickle,
        "n_nans": int(pred_vtime.isna().sum().sum()),
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.head()

Number of pickle files found: 101


,file,shape_rows,shape_cols,same_node_set,same_order,missing_nodes,extra_nodes,n_nans
0,agg_timeseries_000.pkl,54,1570,True,True,0,0,21968
1,agg_timeseries_001.pkl,70,1570,True,True,0,0,44048
2,agg_timeseries_002.pkl,51,1570,True,True,0,0,17319
3,agg_timeseries_003.pkl,53,1570,True,True,0,0,20125
4,agg_timeseries_004.pkl,72,1570,True,True,0,0,47163


In [8]:
print(audit_df["same_node_set"].value_counts(dropna=False))
print(audit_df["same_order"].value_counts(dropna=False))

print("\nRows x Cols counts:")
print(audit_df[["shape_rows", "shape_cols"]].value_counts())

print("\nNaN summary:")
print(audit_df["n_nans"].describe())

same_node_set
True    101
Name: count, dtype: int64
same_order
True    101
Name: count, dtype: int64

Rows x Cols counts:
shape_rows  shape_cols
131         1570          35
52          1570           6
51          1570           5
55          1570           5
70          1570           4
53          1570           4
63          1570           4
54          1570           3
72          1570           3
71          1570           2
50          1570           2
48          1570           2
49          1570           2
56          1570           2
76          1570           2
79          1570           2
80          1570           2
67          1570           2
64          1570           2
68          1570           1
47          1570           1
66          1570           1
75          1570           1
73          1570           1
58          1570           1
90          1570           1
86          1570           1
61          1570           1
69          1570           1
65          15

In [13]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PICKLE_DIR = Path(r"C:\Users\USER\Documents\GitHub\datasets\simbarca\all_agg")
pkl_files = sorted(PICKLE_DIR.glob("*.pkl"))

time_audit_rows = []

for pkl_path in pkl_files:
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    pred_vtime = data["pred_vtime"].copy()
    pred_vtime.columns = pred_vtime.columns.astype(int)

    # Make sure time index is datetime
    pred_vtime.index = pd.to_datetime(pred_vtime.index)
    pred_vtime = pred_vtime.sort_index()

    deltas = pred_vtime.index.to_series().diff().dropna()
    delta_minutes = deltas.dt.total_seconds() / 60.0

    time_audit_rows.append({
        "file": pkl_path.name,
        "n_rows": pred_vtime.shape[0],
        "start_time": pred_vtime.index.min(),
        "end_time": pred_vtime.index.max(),
        "n_unique_times": pred_vtime.index.nunique(),
        "min_delta_min": delta_minutes.min() if len(delta_minutes) > 0 else np.nan,
        "max_delta_min": delta_minutes.max() if len(delta_minutes) > 0 else np.nan,
        "n_non_3min_gaps": int((delta_minutes != 3).sum()) if len(delta_minutes) > 0 else 0,
    })

time_audit_df = pd.DataFrame(time_audit_rows)
time_audit_df.head(10)

,file,n_rows,start_time,end_time,n_unique_times,min_delta_min,max_delta_min,n_non_3min_gaps
0,agg_timeseries_000.pkl,54,2005-05-10 07:48:00,2005-05-10 10:27:00,54,3.0,3.0,0
1,agg_timeseries_001.pkl,70,2005-05-10 07:48:00,2005-05-10 11:15:00,70,3.0,3.0,0
2,agg_timeseries_002.pkl,51,2005-05-10 07:48:00,2005-05-10 10:18:00,51,3.0,3.0,0
3,agg_timeseries_003.pkl,53,2005-05-10 07:48:00,2005-05-10 10:24:00,53,3.0,3.0,0
4,agg_timeseries_004.pkl,72,2005-05-10 07:48:00,2005-05-10 11:21:00,72,3.0,3.0,0
5,agg_timeseries_005.pkl,68,2005-05-10 07:48:00,2005-05-10 11:09:00,68,3.0,3.0,0
6,agg_timeseries_006.pkl,70,2005-05-10 07:48:00,2005-05-10 11:15:00,70,3.0,3.0,0
7,agg_timeseries_007.pkl,52,2005-05-10 07:48:00,2005-05-10 10:21:00,52,3.0,3.0,0
8,agg_timeseries_008.pkl,47,2005-05-10 07:48:00,2005-05-10 10:06:00,47,3.0,3.0,0
9,agg_timeseries_009.pkl,71,2005-05-10 07:48:00,2005-05-10 11:18:00,71,3.0,3.0,0


In [14]:
print("Row count distribution:")
print(time_audit_df["n_rows"].value_counts().sort_index())

print("\nFiles with non-3-minute gaps:")
print((time_audit_df["n_non_3min_gaps"] > 0).sum())

print("\nExample files with issues:")
display(time_audit_df[time_audit_df["n_non_3min_gaps"] > 0].head(10))

Row count distribution:
n_rows
47      1
48      2
49      2
50      2
51      5
52      6
53      4
54      3
55      5
56      2
58      1
61      1
63      4
64      2
65      1
66      1
67      2
68      1
69      1
70      4
71      2
72      3
73      1
75      1
76      2
79      2
80      2
86      1
90      1
97      1
131    35
Name: count, dtype: int64

Files with non-3-minute gaps:
0

Example files with issues:


,file,n_rows,start_time,end_time,n_unique_times,min_delta_min,max_delta_min,n_non_3min_gaps


In [15]:
import pickle
import numpy as np
import pandas as pd

dataset_rows = []

graph_nodes = set(node_order)

for day_idx, pkl_path in enumerate(pkl_files):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    pred_vtime = data["pred_vtime"].copy()
    pred_vtime.columns = pred_vtime.columns.astype(int)

    # Safety check on node set
    pickle_nodes = set(pred_vtime.columns.tolist())
    if pickle_nodes != graph_nodes:
        print(f"Skipping {pkl_path.name} بسبب node mismatch")
        continue

    # Align columns to canonical graph order
    pred_vtime = pred_vtime.loc[:, node_order]

    # Make time index proper and sorted
    pred_vtime.index = pd.to_datetime(pred_vtime.index)
    pred_vtime = pred_vtime.sort_index()

    # Iterate over consecutive rows
    for t in range(len(pred_vtime) - 1):
        ts_curr = pred_vtime.index[t]
        ts_next = pred_vtime.index[t + 1]

        # Keep only true 3-minute transitions
        delta_min = (ts_next - ts_curr).total_seconds() / 60.0
        if delta_min != 3:
            continue

        y_curr = pred_vtime.iloc[t].to_numpy(dtype=np.float32)
        y_next = pred_vtime.iloc[t + 1].to_numpy(dtype=np.float32)

        valid_mask = (~np.isnan(y_curr)) & (~np.isnan(y_next))
        valid_nodes = np.where(valid_mask)[0]

        for node_idx in valid_nodes:
            x_curr = float(y_curr[node_idx])
            y_t1 = float(y_next[node_idx])

            dataset_rows.append({
                "day_idx": day_idx,
                "day_name": pkl_path.name,
                "time_idx_in_file": t,
                "timestamp_t": ts_curr,
                "timestamp_t1": ts_next,
                "node_idx": int(node_idx),
                "node_id": int(node_order[node_idx]),
                "x_curr_vtime": x_curr,
                "y_next_vtime": y_t1,
                "y_resid_vtime": y_t1 - x_curr,
            })

dataset_df = pd.DataFrame(dataset_rows)

print("Dataset shape:", dataset_df.shape)
dataset_df.head()

Dataset shape: (7380162, 10)


,day_idx,day_name,time_idx_in_file,timestamp_t,timestamp_t1,node_idx,node_id,x_curr_vtime,y_next_vtime,y_resid_vtime
0,0,agg_timeseries_000.pkl,0,2005-05-10 07:48:00,2005-05-10 07:51:00,1,513,100.687225,587.592102,486.904877
1,0,agg_timeseries_000.pkl,0,2005-05-10 07:48:00,2005-05-10 07:51:00,2,514,76.256653,137.653290,61.396637
2,0,agg_timeseries_000.pkl,0,2005-05-10 07:48:00,2005-05-10 07:51:00,3,516,404.506958,392.044525,-12.462433
3,0,agg_timeseries_000.pkl,0,2005-05-10 07:48:00,2005-05-10 07:51:00,5,594,139.957565,131.178207,-8.779358
4,0,agg_timeseries_000.pkl,0,2005-05-10 07:48:00,2005-05-10 07:51:00,13,602,34.754341,67.076050,32.321709


In [16]:
print("Number of unique files/days:", dataset_df["day_name"].nunique())
print("Number of unique nodes:", dataset_df["node_id"].nunique())

print("\nTarget summary:")
print(dataset_df["y_next_vtime"].describe())

print("\nResidual summary:")
print(dataset_df["y_resid_vtime"].describe())

print("\nFirst timestamps:")
display(dataset_df[["day_name", "timestamp_t", "timestamp_t1"]].head(10))

Number of unique files/days: 101
Number of unique nodes: 1525

Target summary:
count    7.380162e+06
mean     1.136254e+03
std      2.254889e+03
min      5.642289e-05
25%      1.189035e+02
50%      3.929760e+02
75%      1.086828e+03
max      3.888000e+04
Name: y_next_vtime, dtype: float64

Residual summary:
count    7.380162e+06
mean    -1.722645e+00
std      6.812955e+02
min     -3.859250e+04
25%     -1.090989e+02
50%      0.000000e+00
75%      1.142107e+02
max      1.804894e+04
Name: y_resid_vtime, dtype: float64

First timestamps:


,day_name,timestamp_t,timestamp_t1
0,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
1,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
2,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
3,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
4,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
5,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
6,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
7,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
8,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00
9,agg_timeseries_000.pkl,2005-05-10 07:48:00,2005-05-10 07:51:00


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = dataset_df["y_next_vtime"].values
y_pred = dataset_df["x_curr_vtime"].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Persistence baseline")
print("MAE :", mae)
print("RMSE:", rmse)

Persistence baseline
MAE : 301.4181625342654
RMSE: 681.2976815913464


In [19]:
rng = np.random.default_rng(42)

unique_days = np.array(sorted(dataset_df["day_idx"].unique()))
rng.shuffle(unique_days)

n_days_total = len(unique_days)
n_train = int(0.70 * n_days_total)
n_val = int(0.15 * n_days_total)
n_test = n_days_total - n_train - n_val

train_days = set(unique_days[:n_train])
val_days = set(unique_days[n_train:n_train + n_val])
test_days = set(unique_days[n_train + n_val:])

train_df = dataset_df[dataset_df["day_idx"].isin(train_days)].copy()
val_df = dataset_df[dataset_df["day_idx"].isin(val_days)].copy()
test_df = dataset_df[dataset_df["day_idx"].isin(test_days)].copy()

print("Train rows:", len(train_df), "| days:", train_df["day_idx"].nunique())
print("Val rows  :", len(val_df), "| days:", val_df["day_idx"].nunique())
print("Test rows :", len(test_df), "| days:", test_df["day_idx"].nunique())

Train rows: 5256517 | days: 70
Val rows  : 1099565 | days: 15
Test rows : 1024080 | days: 16


In [20]:
def eval_persistence(df):
    y_true = df["y_next_vtime"].values
    y_pred = df["x_curr_vtime"].values
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    mae, rmse = eval_persistence(split_df)
    print(f"{split_name.upper()} | MAE={mae:.6f} | RMSE={rmse:.6f}")

TRAIN | MAE=307.922154 | RMSE=700.891083
VAL | MAE=316.799518 | RMSE=716.025251
TEST | MAE=251.518601 | RMSE=522.547187
